In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/notebooks/rustemaliyev/resume/__results__.html
/kaggle/input/notebooks/rustemaliyev/resume/__resultx__.html
/kaggle/input/notebooks/rustemaliyev/resume/__notebook__.ipynb
/kaggle/input/notebooks/rustemaliyev/resume/__output__.json
/kaggle/input/notebooks/rustemaliyev/resume/custom.css
/kaggle/input/datasets/adityarajsrv/job-descriptions-2025-tech-and-non-tech-roles/job_dataset.json
/kaggle/input/datasets/adityarajsrv/job-descriptions-2025-tech-and-non-tech-roles/job_dataset.csv
/kaggle/input/datasets/ckshetty/candidate-job-role-dataset/candidate_job_role_dataset.csv


In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

class BatchATSScorer:
    def __init__(self, model_name: str = 'BAAI/bge-small-en-v1.5'):
        """
        Initializes the embedding model.
        """
        print(f"Loading embedding model '{model_name}'...")
        self.model = SentenceTransformer(model_name)

    def load_and_prep_data(self, csv_path: str, limit: int = None) -> pd.DataFrame:
        """
        Loads the Kaggle dataset and combines split columns into one text block.
        """
        print(f"Loading dataset from {csv_path}...")
        df = pd.read_csv(csv_path)
        
        # Replace NaN (empty) values with empty strings so they don't break the text combination
        df = df.fillna('')
        
        # Combine Title, Skills, Responsibilities, and Keywords into a single text column
        print("Stitching job details together...")
        df['combined_text'] = (
            "Title: " + df['Title'] + "\n" +
            "Skills: " + df['Skills'] + "\n" +
            "Responsibilities: " + df['Responsibilities'] + "\n" +
            "Keywords: " + df['Keywords']
        )
        
        # Optional: limit rows for testing purposes to save compute time
        if limit:
            df = df.head(limit)
            
        print(f"Prepared {len(df)} job descriptions.")
        return df

    def compute_bulk_semantic_scores(self, resume_text: str, df: pd.DataFrame, jd_column: str = 'combined_text') -> pd.DataFrame:
        """
        Computes semantic similarity for a resume against an entire DataFrame of JDs.
        """
        print("Encoding candidate resume...")
        resume_vec = self.model.encode(resume_text)

        print("Encoding job descriptions (this may take a moment)...")
        # Extract JD text as a list and encode in batches
        jd_texts = df[jd_column].tolist()
        jd_vecs = self.model.encode(jd_texts, show_progress_bar=True)

        print("Calculating similarity scores...")
        similarity_matrix = cosine_similarity([resume_vec], jd_vecs)
        
        # Flatten the matrix to a 1D array of scores and scale to 100
        semantic_scores = np.maximum(0.0, similarity_matrix[0] * 100)
        
        # Attach scores to the dataframe
        df['semantic_score'] = semantic_scores
        return df

    def calculate_composite_score(self, df: pd.DataFrame, keyword_col: str = 'mock_keyword_score', llm_col: str = 'mock_llm_score') -> pd.DataFrame:
        """
        Applies the 50/30/20 weighted composite scoring.
        """
        df['final_ats_score'] = (
            (df['semantic_score'] * 0.50) + 
            (df[keyword_col] * 0.30) + 
            (df[llm_col] * 0.20)
        )
        return df

# ==========================================
# Kaggle Execution Script
# ==========================================
if __name__ == "__main__":
    # 1. Initialize the scorer
    scorer = BatchATSScorer()

    # 2. Define the Kaggle Dataset Path
    KAGGLE_CSV_PATH = '/kaggle/input/datasets/adityarajsrv/job-descriptions-2025-tech-and-non-tech-roles/job_dataset.csv'

    # Load data (Limiting to 500 for a fast test run)
    try:
        jobs_df = scorer.load_and_prep_data(KAGGLE_CSV_PATH, limit=500)
    except FileNotFoundError:
        print(f"Error: Could not find {KAGGLE_CSV_PATH}.")
        exit()

    # 3. Define the Candidate's Resume
    candidate_resume = """
    Software Engineer with 4 years of experience specializing in Python, machine learning, 
    and data pipelines. Built RESTful APIs using FastAPI and deployed models using Docker 
    and AWS EC2. Proficient in SQL and PostgreSQL.
    """

    # 4. Compute Semantic Scores (using the newly created 'combined_text' column)
    scored_df = scorer.compute_bulk_semantic_scores(candidate_resume, jobs_df, jd_column='combined_text')

    # 5. Mock the Keyword and LLM scores for the sake of the composite formula
    scored_df['mock_keyword_score'] = np.random.uniform(50, 100, len(scored_df))
    scored_df['mock_llm_score'] = np.random.uniform(60, 95, len(scored_df))

    # 6. Calculate Final Score
    final_df = scorer.calculate_composite_score(scored_df)

    # 7. Sort by the highest final ATS score and display the top 5 matches
    top_matches = final_df.sort_values(by='final_ats_score', ascending=False).head(5)
    
    print("\n--- Top 5 Job Matches for Candidate ---")
    
    # We now know 'Title' is the correct column name to display
    display_cols = ['Title', 'ExperienceLevel', 'final_ats_score', 'semantic_score']
    print(top_matches[display_cols].to_string(index=False))

In [ ]:
import pandas as pd

KAGGLE_CSV_PATH = '/kaggle/input/datasets/adityarajsrv/job-descriptions-2025-tech-and-non-tech-roles/job_dataset.csv'
df = pd.read_csv(KAGGLE_CSV_PATH)

print("The columns in this dataset are:")
print(df.columns.tolist())


In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

class RecruiterATSScorer:
    def __init__(self, model_name: str = 'BAAI/bge-small-en-v1.5'):
        """
        Initializes the embedding model.
        """
        print(f"Loading embedding model '{model_name}'...")
        self.model = SentenceTransformer(model_name)

    def load_and_prep_candidates(self, csv_path: str, limit: int = None) -> pd.DataFrame:
        """
        Loads the candidate dataset and dynamically combines all columns into a single resume block.
        """
        print(f"Loading candidate dataset from {csv_path}...")
        df = pd.read_csv(csv_path)
        
        # Replace empty values with empty strings
        df = df.fillna('')
        
        print("Dynamically stitching candidate details together...")
        # This loop goes through every row and stitches all column names and their values together
        # e.g., "Name: John | Skills: Python | Experience: 5 ..."
        df['combined_resume'] = df.apply(
            lambda row: '\n'.join([f"{col}: {str(val)}" for col, val in row.items()]), 
            axis=1
        )
        
        if limit:
            df = df.head(limit)
            
        print(f"Prepared {len(df)} candidate profiles.")
        return df

    def find_top_candidates(self, recruiter_query: str, df: pd.DataFrame) -> pd.DataFrame:
        """
        Takes the recruiter's job post and searches the database of resumes.
        """
        print("Encoding recruiter's job requirements...")
        job_vec = self.model.encode(recruiter_query)

        print("Encoding candidate database (this may take a moment)...")
        resume_texts = df['combined_resume'].tolist()
        resume_vecs = self.model.encode(resume_texts, show_progress_bar=True)

        print("Calculating similarity scores...")
        similarity_matrix = cosine_similarity([job_vec], resume_vecs)
        
        # Extract scores and convert to 0-100 scale
        semantic_scores = np.maximum(0.0, similarity_matrix[0] * 100)
        df['match_score'] = semantic_scores
        
        return df


# ==========================================
# Recruiter Execution Script
# ==========================================
if __name__ == "__main__":
    scorer = RecruiterATSScorer()

    # 1. Define the Recruiter's Input
    job_position = "Senior Backend Developer"
    years_of_experience = "5"
    skillset = "Python, FastAPI, PostgreSQL, AWS, Docker"
    job_description = "We need a senior engineer to design scalable backend APIs, optimize database queries, and manage cloud deployments."

    # Stitch the recruiter's inputs into one cohesive query block
    recruiter_query = f"""
    Title: {job_position}
    Required Experience: {years_of_experience} years
    Skills: {skillset}
    Description: {job_description}
    """

    # 2. Load the actual Kaggle Candidate Dataset
    KAGGLE_CANDIDATE_PATH = '/kaggle/input/datasets/ckshetty/candidate-job-role-dataset/candidate_job_role_dataset.csv'
    
    try:
        # Limiting to 1000 rows for a quick test run. Remove `limit=1000` to process the whole file.
        candidates_df = scorer.load_and_prep_candidates(KAGGLE_CANDIDATE_PATH, limit=1000)
    except FileNotFoundError:
        print(f"Error: Could not find {KAGGLE_CANDIDATE_PATH}. Make sure it is attached to your notebook.")
        exit()

    # 3. Run the search
    scored_candidates = scorer.find_top_candidates(recruiter_query, candidates_df)

    # 4. Sort and Display Top Candidates
    top_resumes = scored_candidates.sort_values(by='match_score', ascending=False).head(5)
    
    print("\n" + "="*50)
    print("TOP 5 CANDIDATES FOR: " + job_position.upper())
    print("="*50)
    
    # We don't know the exact column names for Name or Title in your CSV, 
    # so we will print the top match scores along with the first 100 characters of their profile.
    for index, row in top_resumes.iterrows():
        print(f"Match Score: {row['match_score']:.2f}/100")
        
        # Extracting a snippet of the combined resume for display
        snippet = row['combined_resume'].replace('\n', ' | ')[:120]
        print(f"Profile Snapshot: {snippet}...")
        print("-" * 50)

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

class RecruiterATSScorer:
    def __init__(self, model_name: str = 'BAAI/bge-small-en-v1.5'):
        """
        Initializes the embedding model.
        """
        print(f"Loading embedding model '{model_name}'...")
        self.model = SentenceTransformer(model_name)

    def load_and_prep_candidates(self, csv_path: str, limit: int = None) -> pd.DataFrame:
        """
        Loads the candidate dataset and dynamically combines all columns into a single resume block.
        """
        print(f"\nLoading candidate dataset from {csv_path}...")
        df = pd.read_csv(csv_path)
        
        # Replace empty values with empty strings
        df = df.fillna('')
        
        print("Dynamically stitching candidate details together...")
        df['combined_resume'] = df.apply(
            lambda row: '\n'.join([f"{col}: {str(val)}" for col, val in row.items()]), 
            axis=1
        )
        
        if limit:
            df = df.head(limit)
            
        print(f"Prepared {len(df)} candidate profiles.")
        return df

    def find_top_candidates(self, recruiter_query: str, df: pd.DataFrame) -> pd.DataFrame:
        """
        Takes the recruiter's job post and searches the database of resumes.
        """
        print("\nEncoding recruiter's job requirements...")
        job_vec = self.model.encode(recruiter_query)

        print("Encoding candidate database (this may take a moment)...")
        resume_texts = df['combined_resume'].tolist()
        resume_vecs = self.model.encode(resume_texts, show_progress_bar=True)

        print("Calculating similarity scores...")
        similarity_matrix = cosine_similarity([job_vec], resume_vecs)
        
        # Extract scores and convert to 0-100 scale
        semantic_scores = np.maximum(0.0, similarity_matrix[0] * 100)
        df['match_score'] = semantic_scores
        
        return df


# ==========================================
# Recruiter Execution Script
# ==========================================
if __name__ == "__main__":
    scorer = RecruiterATSScorer()

    print("\n" + "="*50)
    print("📝 RECRUITER INPUT DASHBOARD")
    print("="*50)
    
    # 1. Ask the Recruiter for Input Dynamically
    job_position = input("1. Enter Job Title (e.g., Senior Backend Developer): ")
    years_of_experience = input("2. Enter Required Years of Experience (e.g., 5): ")
    skillset = input("3. Enter Required Skills (e.g., Python, AWS, SQL): ")
    job_description = input("4. Enter a Brief Job Description: ")

    # Stitch the recruiter's inputs into one cohesive query block
    recruiter_query = f"""
    Title: {job_position}
    Required Experience: {years_of_experience} years
    Skills: {skillset}
    Description: {job_description}
    """

    # 2. Load the actual Kaggle Candidate Dataset
    KAGGLE_CANDIDATE_PATH = '/kaggle/input/datasets/ckshetty/candidate-job-role-dataset/candidate_job_role_dataset.csv'
    
    try:
        # Limiting to 1000 rows for a quick test run. Remove `limit=1000` to process the whole file.
        candidates_df = scorer.load_and_prep_candidates(KAGGLE_CANDIDATE_PATH, limit=1000)
    except FileNotFoundError:
        print(f"\nError: Could not find {KAGGLE_CANDIDATE_PATH}. Make sure it is attached to your notebook.")
        exit()

    # 3. Run the search
    scored_candidates = scorer.find_top_candidates(recruiter_query, candidates_df)

    # 4. Sort and Display Top Candidates
    top_resumes = scored_candidates.sort_values(by='match_score', ascending=False).head(5)
    
    print("\n" + "="*50)
    print(f"TOP 5 CANDIDATES FOR: {job_position.upper()}")
    print("="*50)
    
    for index, row in top_resumes.iterrows():
        print(f"Match Score: {row['match_score']:.2f}/100")
        
        # Extracting a snippet of the combined resume for display
        snippet = row['combined_resume'].replace('\n', ' | ')[:120]
        print(f"Profile Snapshot: {snippet}...")
        print("-" * 50)

In [ ]:
"""
Ensemble ATS Scorer  —  5 Scoring Signals
==========================================

Signal          Model / Method                          Weight
──────────────  ──────────────────────────────────────  ──────
BGE Bi-Encoder  BAAI/bge-small-en-v1.5                  25 %
MiniLM Encoder  all-MiniLM-L6-v2                        20 %
Cross-Encoder   cross-encoder/ms-marco-MiniLM-L-6-v2    30 %  ← deep learning reranker
TF-IDF          sklearn TfidfVectorizer (1-2 grams)      15 %
Skill Overlap   Exact token matching                     10 %

Why Cross-Encoder?
  Bi-encoders encode the query and document *independently*, then compare
  the resulting vectors. A cross-encoder feeds BOTH texts through the
  transformer at once, so every token in the resume can attend to every
  token in the job post — it learns richer, context-aware relevance signals
  at the cost of speed (one forward pass per candidate pair).
"""

import re
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ── ensemble weights (must sum to 1.0) ────────────────────────────────────────

WEIGHTS = {
    "bge":          0.25,
    "minilm":       0.20,
    "cross_encoder":0.30,   # deep learning reranker — highest weight
    "tfidf":        0.15,
    "skills":       0.10,
}

assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9, "Weights must sum to 1.0"


# ── helpers ───────────────────────────────────────────────────────────────────

def _extract_skill_tokens(text: str) -> set:
    """Splits comma/slash/newline-delimited skill phrases (1–4 words)."""
    tokens = re.split(r"[,;/\n]+", text.lower())
    return {t.strip() for t in tokens if 1 <= len(t.split()) <= 4 and len(t) > 1}


def _skill_overlap(query_skills: set, resume: str) -> float:
    """Fraction of required skills found verbatim in the resume → [0, 100]."""
    if not query_skills:
        return 0.0
    hit = sum(1 for s in query_skills if s in resume.lower())
    return (hit / len(query_skills)) * 100.0


def _normalise(arr: np.ndarray) -> np.ndarray:
    """Min-max normalise an array to [0, 100]."""
    lo, hi = arr.min(), arr.max()
    if hi == lo:
        return np.full_like(arr, 50.0, dtype=float)
    return ((arr - lo) / (hi - lo)) * 100.0


# ── main class ────────────────────────────────────────────────────────────────

class EnsembleATSScorer:
    """
    Fuses five independent scoring signals — including a deep-learning
    Cross-Encoder reranker — into a single weighted ATS match score.
    """

    def __init__(
        self,
        bge_model:    str = "BAAI/bge-small-en-v1.5",
        mini_model:   str = "sentence-transformers/all-MiniLM-L6-v2",
        cross_model:  str = "cross-encoder/ms-marco-MiniLM-L-6-v2",
    ):
        _banner("ENSEMBLE ATS SCORER  —  loading models")

        print(f"\n  [1/3] BGE bi-encoder     : {bge_model}")
        self.bge = SentenceTransformer(bge_model)

        print(f"  [2/3] MiniLM bi-encoder  : {mini_model}")
        self.minilm = SentenceTransformer(mini_model)

        print(f"  [3/3] Cross-Encoder (DL) : {cross_model}")
        self.cross_encoder = CrossEncoder(
            cross_model,
            max_length=512,
            default_activation_function=None,   # raw logits → we normalise later
        )

        print("\n  ✅  All models loaded.\n")

    # ── data loading ──────────────────────────────────────────────────────────

    def load_and_prep_candidates(
        self, csv_path: str, limit: int = None
    ) -> pd.DataFrame:
        print(f"Loading candidates from: {csv_path}")
        df = pd.read_csv(csv_path).fillna("")
        df["combined_resume"] = df.apply(
            lambda row: "\n".join(f"{c}: {v}" for c, v in row.items()), axis=1
        )
        if limit:
            df = df.head(limit)
        print(f"Prepared {len(df)} profiles.\n")
        return df

    # ── individual signals ────────────────────────────────────────────────────

    def _score_bge(self, query: str, texts: list) -> np.ndarray:
        """Cosine similarity between BGE embeddings."""
        q = self.bge.encode(query)
        r = self.bge.encode(texts, show_progress_bar=True)
        return np.maximum(0.0, cosine_similarity([q], r)[0] * 100)

    def _score_minilm(self, query: str, texts: list) -> np.ndarray:
        """Cosine similarity between MiniLM embeddings."""
        q = self.minilm.encode(query)
        r = self.minilm.encode(texts, show_progress_bar=True)
        return np.maximum(0.0, cosine_similarity([q], r)[0] * 100)

    def _score_cross_encoder(self, query: str, texts: list) -> np.ndarray:
        """
        Deep Learning Cross-Encoder Reranker
        ─────────────────────────────────────
        Feeds each (job_post, resume) pair through a shared transformer so
        that every resume token can attend to every query token — richer
        relevance signals than independent bi-encoder embeddings.

        The model outputs raw logits (higher = more relevant). We normalise
        across the candidate pool so scores land in [0, 100].

        Note: O(N) forward passes — slower than bi-encoders but far more
        expressive. Use batch_size to balance speed vs. GPU memory.
        """
        print("  Running cross-encoder inference (one pass per candidate) …")
        pairs = [(query, t) for t in texts]
        logits = self.cross_encoder.predict(
            pairs,
            batch_size=32,
            show_progress_bar=True,
            convert_to_numpy=True,
        )
        return _normalise(logits.astype(float))

    def _score_tfidf(self, query: str, texts: list) -> np.ndarray:
        """TF-IDF cosine similarity on unigrams + bigrams."""
        corpus = [query] + texts
        mat = TfidfVectorizer(
            ngram_range=(1, 2), stop_words="english", max_features=8_000
        ).fit_transform(corpus)
        return np.maximum(0.0, cosine_similarity(mat[0], mat[1:])[0] * 100)

    def _score_skills(self, query: str, texts: list) -> np.ndarray:
        """Literal skill-token overlap between the job post and each resume."""
        skills = _extract_skill_tokens(query)
        return np.array([_skill_overlap(skills, t) for t in texts])

    # ── ensemble ──────────────────────────────────────────────────────────────

    def find_top_candidates(
        self, recruiter_query: str, df: pd.DataFrame, top_n: int = 5
    ) -> pd.DataFrame:
        """
        Runs all five signals and fuses them into a weighted ensemble score.
        Stores per-signal scores so results are fully explainable.
        """
        texts = df["combined_resume"].tolist()
        W     = WEIGHTS

        print("─" * 60)
        print("[Signal 1/5]  BGE Bi-Encoder …")
        bge_s    = self._score_bge(recruiter_query, texts)

        print("\n[Signal 2/5]  MiniLM Bi-Encoder …")
        mini_s   = self._score_minilm(recruiter_query, texts)

        print("\n[Signal 3/5]  Cross-Encoder (Deep Learning) …")
        cross_s  = self._score_cross_encoder(recruiter_query, texts)

        print("\n[Signal 4/5]  TF-IDF Keyword Match …")
        tfidf_s  = self._score_tfidf(recruiter_query, texts)

        print("\n[Signal 5/5]  Hard Skill Overlap …")
        skill_s  = self._score_skills(recruiter_query, texts)

        ensemble = (
            W["bge"]           * bge_s
            + W["minilm"]      * mini_s
            + W["cross_encoder"] * cross_s
            + W["tfidf"]       * tfidf_s
            + W["skills"]      * skill_s
        )

        df = df.copy()
        df["score_bge"]          = np.round(bge_s,   2)
        df["score_minilm"]       = np.round(mini_s,  2)
        df["score_cross_encoder"]= np.round(cross_s, 2)
        df["score_tfidf"]        = np.round(tfidf_s, 2)
        df["score_skills"]       = np.round(skill_s, 2)
        df["match_score"]        = np.round(ensemble, 2)

        return df.sort_values("match_score", ascending=False)


# ── display helpers ───────────────────────────────────────────────────────────

def _banner(title: str):
    print("\n" + "=" * 60)
    print(f"  {title}")
    print("=" * 60)


def _display_results(top_df: pd.DataFrame, job_position: str):
    _banner(f"TOP {len(top_df)} CANDIDATES — {job_position.upper()}")

    signals = [
        ("score_bge",           f"BGE Bi-Encoder         ({int(WEIGHTS['bge']*100)}%)"),
        ("score_minilm",        f"MiniLM Bi-Encoder      ({int(WEIGHTS['minilm']*100)}%)"),
        ("score_cross_encoder", f"Cross-Encoder [DL]     ({int(WEIGHTS['cross_encoder']*100)}%)"),
        ("score_tfidf",         f"TF-IDF Keywords        ({int(WEIGHTS['tfidf']*100)}%)"),
        ("score_skills",        f"Skill Overlap          ({int(WEIGHTS['skills']*100)}%)"),
        ("match_score",         "ENSEMBLE SCORE"),
    ]

    for rank, (_, row) in enumerate(top_df.iterrows(), start=1):
        print(f"\n{'─'*60}")
        print(f"  RANK #{rank}")
        snippet = row["combined_resume"].replace("\n", " | ")[:150]
        print(f"  Profile : {snippet}…")
        print()
        print(f"  ┌{'─'*38}┬{'─'*11}┐")
        print(f"  │ {'Signal':<36} │  {'Score':>7}  │")
        print(f"  ├{'─'*38}┼{'─'*11}┤")
        for col, label in signals:
            marker = "  ◄" if col == "match_score" else ""
            sep    = f"  ├{'─'*38}┼{'─'*11}┤" if col == "score_skills" else ""
            if sep:
                print(sep)
            print(f"  │ {label:<36} │  {row[col]:>6.2f}   │{marker}")
        print(f"  └{'─'*38}┴{'─'*11}┘")

    print("\n" + "=" * 60)


# ── entry point ───────────────────────────────────────────────────────────────

if __name__ == "__main__":
    scorer = EnsembleATSScorer()

    _banner("📝  RECRUITER INPUT DASHBOARD")
    job_position        = input("1. Job Title (e.g., Senior Backend Developer): ")
    years_of_experience = input("2. Required Years of Experience (e.g., 5):     ")
    skillset            = input("3. Required Skills (e.g., Python, AWS, SQL):   ")
    job_description     = input("4. Brief Job Description:                       ")

    recruiter_query = (
        f"Title: {job_position}\n"
        f"Required Experience: {years_of_experience} years\n"
        f"Skills: {skillset}\n"
        f"Description: {job_description}"
    )

    KAGGLE_CANDIDATE_PATH = (
        "/kaggle/input/datasets/ckshetty/"
        "candidate-job-role-dataset/candidate_job_role_dataset.csv"
    )

    try:
        candidates_df = scorer.load_and_prep_candidates(
            KAGGLE_CANDIDATE_PATH, limit=1000
        )
    except FileNotFoundError:
        print(f"\n❌  File not found: {KAGGLE_CANDIDATE_PATH}")
        exit()

    top_n     = 5
    scored_df = scorer.find_top_candidates(recruiter_query, candidates_df, top_n=top_n)
    _display_results(scored_df.head(top_n), job_position)

In [ ]:
pip install rank-bm25

In [ ]:
"""
Ensemble ATS Scorer  —  5 Signals  (BM25 + Expanded Skill Matching)
=====================================================================

Signal            Model / Method                            Weight
────────────────  ────────────────────────────────────────  ──────
BGE Bi-Encoder    BAAI/bge-small-en-v1.5                    25 %
MiniLM Encoder    all-MiniLM-L6-v2                          20 %
Cross-Encoder     cross-encoder/ms-marco-MiniLM-L-6-v2      30 %  ← deep learning reranker
BM25              rank_bm25 (Okapi BM25)                    15 %  ← replaces TF-IDF
Skill Overlap     Token match + synonym expansion            10 %

Why BM25 instead of TF-IDF?
─────────────────────────────────────────────────────────────────
TF-IDF produces extremely low scores when a SHORT query is matched
against LONG documents because the query terms account for a tiny
fraction of the total vocabulary, making the cosine vector very
sparse.  BM25 (Okapi BM25) fixes this with two key parameters:

  k1  — controls term-frequency saturation (repeated words don't
         keep boosting score linearly).
  b   — controls document-length normalisation (long docs are
         penalised proportionally, so a short query isn't drowned).

Result: BM25 is the industry-standard IR algorithm (used by
Elasticsearch, Solr, Lucene) specifically designed for the
"short query vs. long document" retrieval problem.

Why expanded skill matching?
─────────────────────────────────────────────────────────────────
Exact-match alone misses aliases:
  "ML"  ≠  "machine learning"
  "JS"  ≠  "javascript"
  "k8s" ≠  "kubernetes"

We maintain a synonym table and expand both the query skills and
the resume text before comparing, so abbreviations are caught.
"""

import re
import string
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity

# BM25  —  pip install rank-bm25
try:
    from rank_bm25 import BM25Okapi
except ImportError:
    raise ImportError(
        "BM25 requires `rank-bm25`.  Install with:  pip install rank-bm25"
    )


# ── ensemble weights (must sum to 1.0) ────────────────────────────────────────

WEIGHTS = {
    "bge":           0.25,
    "minilm":        0.20,
    "cross_encoder": 0.30,
    "bm25":          0.15,   # ← was tfidf
    "skills":        0.10,
}

assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9, "Weights must sum to 1.0"


# ── skill synonym table ────────────────────────────────────────────────────────
# Format:  canonical_name → [aliases …]
# Both canonical names AND aliases are expanded during matching.

SKILL_SYNONYMS: dict[str, list[str]] = {
    # Programming languages
    "python":             ["py"],
    "javascript":         ["js", "node.js", "nodejs", "node"],
    "typescript":         ["ts"],
    "golang":             ["go"],
    "c++":                ["cpp", "c plus plus"],
    "c#":                 ["csharp", "dotnet", ".net"],
    "ruby on rails":      ["rails", "ror"],
    # ML / AI
    "machine learning":   ["ml", "statistical learning"],
    "deep learning":      ["dl", "neural networks", "neural network"],
    "natural language processing": ["nlp", "text mining"],
    "computer vision":    ["cv", "image recognition"],
    "large language models": ["llm", "llms", "generative ai", "gen ai"],
    "reinforcement learning": ["rl"],
    # Cloud & DevOps
    "amazon web services": ["aws", "amazon cloud"],
    "google cloud platform": ["gcp", "google cloud"],
    "microsoft azure":    ["azure"],
    "kubernetes":         ["k8s", "kube"],
    "docker":             ["containerization", "containers"],
    "continuous integration": ["ci", "ci/cd", "cicd"],
    "continuous deployment": ["cd", "ci/cd", "cicd"],
    "infrastructure as code": ["iac", "terraform", "pulumi"],
    # Databases
    "postgresql":         ["postgres", "psql"],
    "microsoft sql server": ["mssql", "sql server"],
    "mongodb":            ["mongo"],
    "elasticsearch":      ["elastic", "es"],
    "apache kafka":       ["kafka"],
    "apache spark":       ["spark", "pyspark"],
    # Web / frontend
    "react":              ["react.js", "reactjs"],
    "angular":            ["angular.js", "angularjs"],
    "vue":                ["vue.js", "vuejs"],
    "graphql":            ["graph ql"],
    # Data
    "data science":       ["ds", "data scientist"],
    "data engineering":   ["data engineer", "de"],
    "business intelligence": ["bi", "business analytics"],
    "etl":                ["extract transform load", "data pipeline"],
    # Soft / misc
    "agile":              ["scrum", "kanban", "sprint"],
    "rest api":           ["restful", "rest", "api development"],
    "microservices":      ["micro services", "service oriented architecture", "soa"],
}

# Build reverse lookup:  alias → canonical
_ALIAS_TO_CANONICAL: dict[str, str] = {}
for canonical, aliases in SKILL_SYNONYMS.items():
    for alias in aliases:
        _ALIAS_TO_CANONICAL[alias] = canonical


def _expand_text(text: str) -> str:
    """
    Replace known aliases in a text with their canonical form so that
    'AWS' and 'Amazon Web Services' both map to the same token.
    """
    text_lower = text.lower()
    for alias, canonical in _ALIAS_TO_CANONICAL.items():
        # whole-word replacement only (avoid 'go' matching inside 'google')
        text_lower = re.sub(
            rf"\b{re.escape(alias)}\b", canonical, text_lower
        )
    return text_lower


def _tokenise(text: str) -> list[str]:
    """Lowercase, expand synonyms, strip punctuation, split on whitespace."""
    expanded = _expand_text(text)
    expanded = expanded.translate(str.maketrans("", "", string.punctuation))
    return expanded.split()


def _extract_skill_tokens(text: str) -> set:
    """Extract 1-to-4-word skill phrases from comma/slash/newline-delimited text."""
    expanded = _expand_text(text)
    tokens   = re.split(r"[,;/\n]+", expanded)
    return {t.strip() for t in tokens if 1 <= len(t.split()) <= 4 and len(t) > 1}


def _skill_overlap(query_skills: set, resume: str) -> float:
    """
    Fraction of (synonym-expanded) required skills found in the resume.
    Returns a score in [0, 100].
    """
    if not query_skills:
        return 0.0
    resume_expanded = _expand_text(resume)
    hit = sum(1 for s in query_skills if s in resume_expanded)
    return (hit / len(query_skills)) * 100.0


def _normalise(arr: np.ndarray) -> np.ndarray:
    """Min-max normalise an array to [0, 100]."""
    lo, hi = float(arr.min()), float(arr.max())
    if hi == lo:
        return np.full(len(arr), 50.0)
    return ((arr - lo) / (hi - lo)) * 100.0


# ── main class ────────────────────────────────────────────────────────────────

class EnsembleATSScorer:
    """
    Fuses five independent scoring signals — including a Cross-Encoder
    deep-learning reranker and BM25 — into a single weighted ATS score.
    """

    def __init__(
        self,
        bge_model:   str = "BAAI/bge-small-en-v1.5",
        mini_model:  str = "sentence-transformers/all-MiniLM-L6-v2",
        cross_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2",
    ):
        _banner("ENSEMBLE ATS SCORER  —  loading models")

        print(f"\n  [1/3] BGE bi-encoder     : {bge_model}")
        self.bge = SentenceTransformer(bge_model)

        print(f"  [2/3] MiniLM bi-encoder  : {mini_model}")
        self.minilm = SentenceTransformer(mini_model)

        print(f"  [3/3] Cross-Encoder (DL) : {cross_model}")
        self.cross_encoder = CrossEncoder(
            cross_model,
            max_length=512,
            default_activation_function=None,
        )

        print("\n  ✅  All models loaded.\n")

    # ── data loading ──────────────────────────────────────────────────────────

    def load_and_prep_candidates(
        self, csv_path: str, limit: int = None
    ) -> pd.DataFrame:
        print(f"Loading candidates from: {csv_path}")
        df = pd.read_csv(csv_path).fillna("")
        df["combined_resume"] = df.apply(
            lambda row: "\n".join(f"{c}: {v}" for c, v in row.items()), axis=1
        )
        if limit:
            df = df.head(limit)
        print(f"Prepared {len(df)} profiles.\n")
        return df

    # ── individual signals ────────────────────────────────────────────────────

    def _score_bge(self, query: str, texts: list) -> np.ndarray:
        q = self.bge.encode(query)
        r = self.bge.encode(texts, show_progress_bar=True)
        return np.maximum(0.0, cosine_similarity([q], r)[0] * 100)

    def _score_minilm(self, query: str, texts: list) -> np.ndarray:
        q = self.minilm.encode(query)
        r = self.minilm.encode(texts, show_progress_bar=True)
        return np.maximum(0.0, cosine_similarity([q], r)[0] * 100)

    def _score_cross_encoder(self, query: str, texts: list) -> np.ndarray:
        """
        Cross-Encoder: runs (query, resume) pairs through a shared transformer
        with full cross-attention. Raw logits are min-max normalised to [0, 100].
        """
        print("  Running cross-encoder inference …")
        pairs  = [(query, t) for t in texts]
        logits = self.cross_encoder.predict(
            pairs, batch_size=32, show_progress_bar=True, convert_to_numpy=True
        )
        return _normalise(logits.astype(float))

    def _score_bm25(self, query: str, texts: list) -> np.ndarray:
        """
        BM25 (Okapi BM25) — purpose-built for short-query / long-document IR.

        How it differs from TF-IDF:
        ─────────────────────────────────────────────────────────────────────
        TF-IDF score grows linearly with term frequency and is heavily
        affected by document length. When the query is short (50 words)
        and documents are long (500+ words), cosine similarity collapses
        because the shared vocabulary is a tiny fraction of each doc's
        full term vector → scores like 12/100.

        BM25 fixes both problems:
          • Saturation  (k1=1.5): term-frequency contribution plateaus, so
            one mention of "Python" and ten mentions give similar score.
          • Length norm (b=0.75): longer documents are penalised so a
            short query isn't drowned in irrelevant content.

        Both query tokens AND resume tokens are synonym-expanded first so
        'ML' and 'machine learning' score identically.
        """
        tokenised_corpus = [_tokenise(t) for t in texts]
        bm25             = BM25Okapi(tokenised_corpus, k1=1.5, b=0.75)
        query_tokens     = _tokenise(query)
        raw_scores       = np.array(bm25.get_scores(query_tokens), dtype=float)
        return _normalise(raw_scores)

    def _score_skills(self, query: str, texts: list) -> np.ndarray:
        """Synonym-expanded skill-token overlap."""
        skills = _extract_skill_tokens(query)
        return np.array([_skill_overlap(skills, t) for t in texts])

    # ── ensemble ──────────────────────────────────────────────────────────────

    def find_top_candidates(
        self, recruiter_query: str, df: pd.DataFrame, top_n: int = 5
    ) -> pd.DataFrame:
        texts = df["combined_resume"].tolist()
        W     = WEIGHTS

        print("─" * 60)
        print("[Signal 1/5]  BGE Bi-Encoder …")
        bge_s   = self._score_bge(recruiter_query, texts)

        print("\n[Signal 2/5]  MiniLM Bi-Encoder …")
        mini_s  = self._score_minilm(recruiter_query, texts)

        print("\n[Signal 3/5]  Cross-Encoder (Deep Learning) …")
        cross_s = self._score_cross_encoder(recruiter_query, texts)

        print("\n[Signal 4/5]  BM25 Keyword Retrieval …")
        bm25_s  = self._score_bm25(recruiter_query, texts)

        print("\n[Signal 5/5]  Expanded Skill Overlap …")
        skill_s = self._score_skills(recruiter_query, texts)

        ensemble = (
            W["bge"]           * bge_s
            + W["minilm"]      * mini_s
            + W["cross_encoder"] * cross_s
            + W["bm25"]        * bm25_s
            + W["skills"]      * skill_s
        )

        df = df.copy()
        df["score_bge"]          = np.round(bge_s,   2)
        df["score_minilm"]       = np.round(mini_s,  2)
        df["score_cross_encoder"]= np.round(cross_s, 2)
        df["score_bm25"]         = np.round(bm25_s,  2)
        df["score_skills"]       = np.round(skill_s, 2)
        df["match_score"]        = np.round(ensemble, 2)

        return df.sort_values("match_score", ascending=False)


# ── display helpers ───────────────────────────────────────────────────────────

def _banner(title: str):
    print("\n" + "=" * 60)
    print(f"  {title}")
    print("=" * 60)


def _display_results(top_df: pd.DataFrame, job_position: str):
    _banner(f"TOP {len(top_df)} CANDIDATES — {job_position.upper()}")

    signals = [
        ("score_bge",           f"BGE Bi-Encoder         ({int(WEIGHTS['bge']*100)}%)"),
        ("score_minilm",        f"MiniLM Bi-Encoder      ({int(WEIGHTS['minilm']*100)}%)"),
        ("score_cross_encoder", f"Cross-Encoder [DL]     ({int(WEIGHTS['cross_encoder']*100)}%)"),
        ("score_bm25",          f"BM25 Retrieval         ({int(WEIGHTS['bm25']*100)}%)"),
        ("score_skills",        f"Skill Overlap (expanded)({int(WEIGHTS['skills']*100)}%)"),
        ("match_score",         "ENSEMBLE SCORE"),
    ]

    for rank, (_, row) in enumerate(top_df.iterrows(), start=1):
        print(f"\n{'─'*62}")
        print(f"  RANK #{rank}")
        snippet = row["combined_resume"].replace("\n", " | ")[:150]
        print(f"  Profile : {snippet}…")
        print()
        print(f"  ┌{'─'*40}┬{'─'*11}┐")
        print(f"  │ {'Signal':<38} │  {'Score':>7}  │")
        print(f"  ├{'─'*40}┼{'─'*11}┤")
        for col, label in signals:
            marker = "  ◄" if col == "match_score" else ""
            if col == "score_skills":
                print(f"  ├{'─'*40}┼{'─'*11}┤")
            print(f"  │ {label:<38} │  {row[col]:>6.2f}   │{marker}")
        print(f"  └{'─'*40}┴{'─'*11}┘")

    print("\n" + "=" * 62)


# ── entry point ───────────────────────────────────────────────────────────────

if __name__ == "__main__":
    scorer = EnsembleATSScorer()

    _banner("📝  RECRUITER INPUT DASHBOARD")
    job_position        = input("1. Job Title (e.g., Senior Backend Developer): ")
    years_of_experience = input("2. Required Years of Experience (e.g., 5):     ")
    skillset            = input("3. Required Skills (e.g., Python, AWS, SQL):   ")
    job_description     = input("4. Brief Job Description:                       ")

    recruiter_query = (
        f"Title: {job_position}\n"
        f"Required Experience: {years_of_experience} years\n"
        f"Skills: {skillset}\n"
        f"Description: {job_description}"
    )

    KAGGLE_CANDIDATE_PATH = (
        "/kaggle/input/datasets/ckshetty/"
        "candidate-job-role-dataset/candidate_job_role_dataset.csv"
    )

    try:
        candidates_df = scorer.load_and_prep_candidates(
            KAGGLE_CANDIDATE_PATH, limit=1000
        )
    except FileNotFoundError:
        print(f"\n❌  File not found: {KAGGLE_CANDIDATE_PATH}")
        exit()

    top_n     = 5
    scored_df = scorer.find_top_candidates(recruiter_query, candidates_df, top_n=top_n)
    _display_results(scored_df.head(top_n), job_position)

In [ ]:
"""
Industry-Grade Ensemble ATS Scorer  (Bug-Fix Release)
======================================================
Fixes applied vs. previous version:

  BUG 1 — NER Skill Overlap always 0
    Cause : EntityRuler used plain lowercase string patterns, which spaCy
            matches case-sensitively at the token level. "Python" in a resume
            never matched the pattern "python".
    Fix   : Every pattern now uses the token-level dict format
            [{"LOWER": token}, …] which is case-insensitive by design.

  BUG 2 — Temporal Recency always 0
    Cause : Most ATS CSV datasets don't embed "2019–2022" date ranges inside
            free-text skill descriptions. When no year is found near a skill,
            the old code returned 0 — penalising candidates unfairly.
    Fix   : When no date context is found for a skill the score is 50.0
            (neutral / unknown) not 0. Only *confirmed* stale skills are
            penalised. Also detects "X years of experience" phrases as a
            proxy for recency.

  BUG 3 — Experience Multiplier always 0.50 (penalised)
    Cause : Regex r"(\\d{1,2})\\+?\\s*(?:years?|yrs?)" missed common formats
            such as "Experience: 5", "total experience 8 years", or a numeric
            CSV column named 'years_experience' / 'experience'.
    Fix   : (a) Broader regex covering more natural-language patterns.
            (b) Direct numeric-column scan of the original CSV row.
            (c) When no experience data is found at all, use 0.75 (soft
                neutral) instead of the full 0.50 penalty.

Scoring Architecture (unchanged)
─────────────────────────────────────────────────────────────────────
BUCKET          SIGNALS                         BUCKET WT  SIGNAL WT
──────────────  ──────────────────────────────  ─────────  ─────────
Semantic        BGE Bi-Encoder                    45 %       22.5 %
                MiniLM Bi-Encoder                            22.5 %
Precision       Cross-Encoder (DL, sigmoid)       45 %       25.0 %
                BM25 Okapi                                   20.0 %
Validation      NER Skill Overlap                 10 %        6.0 %
                Temporal Recency                              4.0 %

Pre-filter multiplier (applied after ensemble):
  years found >= required  →  × 1.00
  years found <  required  →  × 0.50
  years not detectable     →  × 0.75  (soft neutral — benefit of the doubt)
"""

import re
import string
import datetime
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity

try:
    from rank_bm25 import BM25Okapi
except ImportError:
    raise ImportError("pip install rank-bm25")

try:
    import spacy
except ImportError:
    raise ImportError("pip install spacy && python -m spacy download en_core_web_sm")


# ══════════════════════════════════════════════════════════════════════════════
#  CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

CURRENT_YEAR    = datetime.datetime.now().year
HALF_LIFE_YEARS = 5       # recency half-life in years
HARD_FILTER_PENALTY  = 0.50   # multiplier when years < required
SOFT_NEUTRAL_MULT    = 0.75   # multiplier when years can't be determined

WEIGHTS = {
    "bge":           0.225,
    "minilm":        0.225,
    "cross_encoder": 0.250,
    "bm25":          0.200,
    "ner_skills":    0.060,
    "temporal":      0.040,
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9


# ══════════════════════════════════════════════════════════════════════════════
#  SKILL SYNONYM TABLE
# ══════════════════════════════════════════════════════════════════════════════

SKILL_SYNONYMS: dict[str, list[str]] = {
    "python":                      ["py"],
    "javascript":                  ["js", "node.js", "nodejs", "node"],
    "typescript":                  ["ts"],
    "golang":                      ["go lang", "go"],
    "c++":                         ["cpp", "c plus plus"],
    "c#":                          ["csharp", "dotnet", ".net"],
    "ruby on rails":               ["rails", "ror"],
    "machine learning":            ["ml", "statistical learning"],
    "deep learning":               ["dl", "neural networks", "neural network"],
    "natural language processing": ["nlp", "text mining"],
    "computer vision":             ["cv", "image recognition"],
    "large language models":       ["llm", "llms", "generative ai", "gen ai"],
    "reinforcement learning":      ["rl"],
    "amazon web services":         ["aws", "amazon cloud"],
    "google cloud platform":       ["gcp", "google cloud"],
    "microsoft azure":             ["azure"],
    "kubernetes":                  ["k8s", "kube"],
    "docker":                      ["containerization", "containers"],
    "continuous integration":      ["ci", "ci/cd", "cicd"],
    "continuous deployment":       ["cd", "ci/cd", "cicd"],
    "infrastructure as code":      ["iac", "terraform", "pulumi"],
    "postgresql":                  ["postgres", "psql"],
    "microsoft sql server":        ["mssql", "sql server"],
    "mongodb":                     ["mongo"],
    "elasticsearch":               ["elastic", "es"],
    "apache kafka":                ["kafka"],
    "apache spark":                ["spark", "pyspark"],
    "react":                       ["react.js", "reactjs"],
    "angular":                     ["angular.js", "angularjs"],
    "vue":                         ["vue.js", "vuejs"],
    "graphql":                     ["graph ql"],
    "data science":                ["ds", "data scientist"],
    "data engineering":            ["data engineer", "de"],
    "business intelligence":       ["bi", "business analytics"],
    "etl":                         ["extract transform load", "data pipeline"],
    "agile":                       ["scrum", "kanban", "sprint"],
    "rest api":                    ["restful", "rest", "api development"],
    "microservices":               ["micro services", "soa"],
    "artificial intelligence":     ["ai"],
    "sql":                         ["structured query language"],
    "java":                        ["java ee", "java se", "spring", "spring boot"],
    "android":                     ["android development", "android sdk"],
    "ios":                         ["swift", "objective-c", "xcode"],
    "flutter":                     ["dart"],
    "devops":                      ["dev ops", "site reliability", "sre"],
}

_ALIAS_TO_CANONICAL: dict[str, str] = {
    alias: canonical
    for canonical, aliases in SKILL_SYNONYMS.items()
    for alias in aliases
}


# ══════════════════════════════════════════════════════════════════════════════
#  NER PIPELINE  (FIX 1 — case-insensitive token patterns)
# ══════════════════════════════════════════════════════════════════════════════

def _make_ner_patterns() -> list[dict]:
    """
    Build EntityRuler patterns using the token-level dict format:
        [{"LOWER": "machine"}, {"LOWER": "learning"}]
    spaCy applies LOWER matching case-insensitively, so "Python",
    "PYTHON", and "python" all match the same pattern.
    """
    patterns = []
    seen     = set()
    for canonical, aliases in SKILL_SYNONYMS.items():
        for surface in [canonical] + aliases:
            if surface in seen:
                continue
            seen.add(surface)
            tokens = surface.split()
            patterns.append({
                "label":   "TECH_SKILL",
                "pattern": [{"LOWER": t} for t in tokens],
            })
    return patterns


def _build_ner_pipeline():
    nlp   = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])
    ruler = nlp.add_pipe("entity_ruler", config={"overwrite_ents": True})
    ruler.add_patterns(_make_ner_patterns())
    return nlp


def _extract_ner_skills(nlp, text: str) -> set:
    doc = nlp(text[:50_000])
    return {ent.text.lower() for ent in doc.ents if ent.label_ == "TECH_SKILL"}


def _score_ner_skills(nlp, query_skills: set, texts: list) -> np.ndarray:
    if not query_skills:
        return np.full(len(texts), 50.0)
    scores = []
    for text in texts:
        resume_skills = _extract_ner_skills(nlp, text)
        hit = len(query_skills & resume_skills)
        scores.append((hit / len(query_skills)) * 100.0)
    return np.array(scores)


# ══════════════════════════════════════════════════════════════════════════════
#  TEMPORAL RECENCY  (FIX 2 — neutral fallback when no dates found)
# ══════════════════════════════════════════════════════════════════════════════

_YEAR_RANGE_RE = re.compile(
    r"(\d{4})\s*[-–—to]+\s*(\d{4}|present|now|current|date)",
    re.IGNORECASE,
)
_SINCE_RE  = re.compile(r"since\s+(\d{4})", re.IGNORECASE)
_SINGLE_YR = re.compile(r"\b(20\d{2}|19\d{2})\b")


def _latest_year_near_skill(skill: str, resume: str) -> int | None:
    """
    Searches ±120 chars around every occurrence of the skill for a year.
    Returns the most recent valid year, or None if nothing found.
    """
    lower     = resume.lower()
    skill_low = skill.lower()
    positions = [m.start() for m in re.finditer(re.escape(skill_low), lower)]
    if not positions:
        return None

    best = None
    for pos in positions:
        window = lower[max(0, pos - 120): pos + 120]

        # "2019–2023" or "2019 – present"
        for m in _YEAR_RANGE_RE.finditer(window):
            end_raw = m.group(2)
            year    = CURRENT_YEAR if not end_raw.isdigit() else int(end_raw)
            if 1990 <= year <= CURRENT_YEAR:
                best = year if best is None else max(best, year)

        # "since 2021"
        for m in _SINCE_RE.finditer(window):
            year = CURRENT_YEAR          # "since X" implies still ongoing
            best = year if best is None else max(best, year)

        # any lone 4-digit year
        for m in _SINGLE_YR.finditer(window):
            year = int(m.group(1))
            if 1990 <= year <= CURRENT_YEAR:
                best = year if best is None else max(best, year)

    return best


def _recency_score(last_year: int | None) -> float:
    """
    FIX: Returns 50.0 (neutral) when no date is found, not 0.
    Only confirmed stale skills are penalised.

    With HALF_LIFE_YEARS = 5:
        last_year == CURRENT_YEAR      →  100.0
        last_year == CURRENT_YEAR - 5  →   50.0
        last_year == CURRENT_YEAR - 10 →   25.0
        last_year is None              →   50.0  ← neutral, not 0
    """
    if last_year is None:
        return 50.0                          # ← FIX: benefit of the doubt
    staleness = max(0, CURRENT_YEAR - last_year)
    return 100.0 * (0.5 ** (staleness / HALF_LIFE_YEARS))


def _score_temporal(query_skills: set, texts: list) -> np.ndarray:
    if not query_skills:
        return np.full(len(texts), 50.0)
    scores = []
    for text in texts:
        per_skill = [
            _recency_score(_latest_year_near_skill(s, text))
            for s in query_skills
        ]
        scores.append(float(np.mean(per_skill)))
    return np.array(scores)


# ══════════════════════════════════════════════════════════════════════════════
#  HARD-REQUIREMENT PRE-FILTER  (FIX 3 — broadened detection + soft neutral)
# ══════════════════════════════════════════════════════════════════════════════

# Common column names that directly store years of experience as a number
_EXP_COL_NAMES = {
    "years_experience", "years of experience", "experience", "experience_years",
    "total_experience", "work_experience", "yrs_experience", "exp_years",
    "years", "exp",
}

# Regex patterns covering a wide range of natural-language formats:
#   "5 years"  |  "5+ years"  |  "5 yrs"
#   "5 years of experience"
#   "experience: 5"  |  "total experience 8 years"
#   "8-year career"
_EXP_TEXT_PATTERNS = [
    re.compile(r"(\d{1,2})\+?\s*(?:years?|yrs?)\s+(?:of\s+)?experience", re.I),
    re.compile(r"experience\s*[:\-]?\s*(\d{1,2})\+?\s*(?:years?|yrs?)?", re.I),
    re.compile(r"(\d{1,2})\+?\s*(?:years?|yrs?)", re.I),
    re.compile(r"(\d{1,2})\s*[-–]\s*year\s+(?:career|experience|professional)", re.I),
    re.compile(r"over\s+(\d{1,2})\s+years?", re.I),
    re.compile(r"more\s+than\s+(\d{1,2})\s+years?", re.I),
]


def _extract_max_years(row: pd.Series) -> int | None:
    """
    Two-pass extraction:
      Pass 1 — check known numeric columns in the CSV row directly.
      Pass 2 — regex scan of the combined_resume text.
    Returns the maximum years found, or None if nothing detected.
    """
    # Pass 1: numeric column scan
    for col in row.index:
        if col.lower().replace(" ", "_") in _EXP_COL_NAMES:
            try:
                val = float(row[col])
                if 0 < val < 60:
                    return int(val)
            except (ValueError, TypeError):
                pass

    # Pass 2: regex across resume text
    resume = str(row.get("combined_resume", ""))
    found  = []
    for pattern in _EXP_TEXT_PATTERNS:
        for m in pattern.finditer(resume):
            try:
                found.append(int(m.group(1)))
            except (IndexError, ValueError):
                pass

    return max(found) if found else None


def _hard_filter_multipliers(df: pd.DataFrame, min_years: int) -> np.ndarray:
    """
    Returns a per-candidate multiplier array:
      years >= min_years  →  1.00   (pass)
      years <  min_years  →  0.50   (hard fail)
      years not found     →  0.75   (soft neutral — FIX)

    Also prints a diagnostic summary.
    """
    multipliers = []
    detected    = {"pass": 0, "fail": 0, "unknown": 0}

    for _, row in df.iterrows():
        max_yrs = _extract_max_years(row)
        if max_yrs is None:
            mult = SOFT_NEUTRAL_MULT
            detected["unknown"] += 1
        elif max_yrs >= min_years:
            mult = 1.00
            detected["pass"] += 1
        else:
            mult = HARD_FILTER_PENALTY
            detected["fail"] += 1
        multipliers.append(mult)

    print(
        f"  Pre-filter summary — "
        f"✅ pass: {detected['pass']}  "
        f"❌ fail: {detected['fail']}  "
        f"❓ unknown: {detected['unknown']}"
    )
    return np.array(multipliers)


# ══════════════════════════════════════════════════════════════════════════════
#  SHARED HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def _expand_text(text: str) -> str:
    t = text.lower()
    for alias, canonical in _ALIAS_TO_CANONICAL.items():
        t = re.sub(rf"\b{re.escape(alias)}\b", canonical, t)
    return t


def _tokenise(text: str) -> list[str]:
    expanded = _expand_text(text)
    return expanded.translate(str.maketrans("", "", string.punctuation)).split()


def _normalise(arr: np.ndarray) -> np.ndarray:
    lo, hi = float(arr.min()), float(arr.max())
    if hi == lo:
        return np.full(len(arr), 50.0)
    return ((arr - lo) / (hi - lo)) * 100.0


def _sigmoid_scale(logits: np.ndarray) -> np.ndarray:
    return (1.0 / (1.0 + np.exp(-logits.astype(float)))) * 100.0


# ══════════════════════════════════════════════════════════════════════════════
#  DIAGNOSTIC HELPER  (run this to debug zero scores)
# ══════════════════════════════════════════════════════════════════════════════

def diagnose(scorer, query: str, sample_resume: str):
    """
    Call this to debug zero scores for a single resume.
    Usage:
        diagnose(scorer, recruiter_query, candidates_df.iloc[0]['combined_resume'])
    """
    print("\n" + "═"*60)
    print("  DIAGNOSTIC REPORT")
    print("═"*60)

    q_skills = _extract_ner_skills(scorer.nlp, query)
    r_skills = _extract_ner_skills(scorer.nlp, sample_resume)
    overlap  = q_skills & r_skills

    print(f"\n  NER — Query skills found   : {sorted(q_skills) or '⚠ NONE'}")
    print(f"  NER — Resume skills found  : {sorted(r_skills) or '⚠ NONE'}")
    print(f"  NER — Overlap              : {sorted(overlap) or '⚠ NONE'}")

    print("\n  Temporal — per query skill:")
    for s in sorted(q_skills):
        yr = _latest_year_near_skill(s, sample_resume)
        sc = _recency_score(yr)
        print(f"    {s:<30} last_year={yr}  recency={sc:.1f}")

    print("\n  Experience extraction:")
    # Build a fake row to call _extract_max_years
    fake_row = pd.Series({"combined_resume": sample_resume})
    yrs = _extract_max_years(fake_row)
    print(f"    Max years detected: {yrs}")

    print("\n" + "═"*60)


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN SCORER CLASS
# ══════════════════════════════════════════════════════════════════════════════

class IndustryATSScorer:

    def __init__(
        self,
        bge_model:   str = "BAAI/bge-small-en-v1.5",
        mini_model:  str = "sentence-transformers/all-MiniLM-L6-v2",
        cross_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2",
    ):
        _banner("INDUSTRY-GRADE ATS SCORER — initialising")

        print(f"\n  [1/4] BGE bi-encoder     : {bge_model}")
        self.bge    = SentenceTransformer(bge_model)

        print(f"  [2/4] MiniLM bi-encoder  : {mini_model}")
        self.minilm = SentenceTransformer(mini_model)

        print(f"  [3/4] Cross-Encoder (DL) : {cross_model}")
        self.cross  = CrossEncoder(
            cross_model, max_length=512, default_activation_function=None
        )

        print("  [4/4] spaCy NER pipeline (case-insensitive patterns) …")
        self.nlp = _build_ner_pipeline()

        print("\n  ✅  All components ready.\n")

    def load_and_prep_candidates(
        self, csv_path: str, limit: int = None
    ) -> pd.DataFrame:
        print(f"Loading candidates from: {csv_path}")
        df = pd.read_csv(csv_path).fillna("")
        df["combined_resume"] = df.apply(
            lambda row: "\n".join(f"{c}: {v}" for c, v in row.items()), axis=1
        )
        if limit:
            df = df.head(limit)

        # Print column names to help debug experience extraction
        exp_cols = [c for c in df.columns if c.lower().replace(" ", "_") in _EXP_COL_NAMES]
        print(f"Prepared {len(df)} profiles.  Experience columns detected: {exp_cols or 'none — will use regex'}\n")
        return df

    def _score_bge(self, query, texts):
        q = self.bge.encode(query)
        r = self.bge.encode(texts, show_progress_bar=True)
        return np.maximum(0.0, cosine_similarity([q], r)[0] * 100)

    def _score_minilm(self, query, texts):
        q = self.minilm.encode(query)
        r = self.minilm.encode(texts, show_progress_bar=True)
        return np.maximum(0.0, cosine_similarity([q], r)[0] * 100)

    def _score_cross_encoder(self, query, texts):
        pairs  = [(query, t) for t in texts]
        logits = self.cross.predict(
            pairs, batch_size=32, show_progress_bar=True, convert_to_numpy=True
        )
        return _sigmoid_scale(logits)

    def _score_bm25(self, query, texts):
        tokenised = [_tokenise(t) for t in texts]
        bm25      = BM25Okapi(tokenised, k1=1.5, b=0.75)
        raw       = np.array(bm25.get_scores(_tokenise(query)), dtype=float)
        return _normalise(raw)

    def find_top_candidates(
        self,
        recruiter_query: str,
        df: pd.DataFrame,
        min_years_exp: int = 0,
        top_n: int = 5,
        run_diagnostic: bool = False,
    ) -> pd.DataFrame:

        texts        = df["combined_resume"].tolist()
        query_skills = _extract_ner_skills(self.nlp, recruiter_query)
        print(f"  NER extracted query skills: {sorted(query_skills) or '⚠ none — check synonym table'}\n")

        if run_diagnostic and len(texts) > 0:
            diagnose(self, recruiter_query, texts[0])

        print("─" * 62)
        print("[Semantic  1/2]  BGE Bi-Encoder …")
        bge_s   = self._score_bge(recruiter_query, texts)

        print("\n[Semantic  2/2]  MiniLM Bi-Encoder …")
        mini_s  = self._score_minilm(recruiter_query, texts)

        print("\n[Precision 1/2]  Cross-Encoder (sigmoid) …")
        cross_s = self._score_cross_encoder(recruiter_query, texts)

        print("\n[Precision 2/2]  BM25 Retrieval …")
        bm25_s  = self._score_bm25(recruiter_query, texts)

        print("\n[Validation 1/2]  NER Skill Overlap …")
        ner_s   = _score_ner_skills(self.nlp, query_skills, texts)

        print("\n[Validation 2/2]  Temporal Recency …")
        temp_s  = _score_temporal(query_skills, texts)

        W        = WEIGHTS
        ensemble = (
            W["bge"]           * bge_s
            + W["minilm"]      * mini_s
            + W["cross_encoder"] * cross_s
            + W["bm25"]        * bm25_s
            + W["ner_skills"]  * ner_s
            + W["temporal"]    * temp_s
        )

        print(f"\n[Pre-filter]  Checking ≥ {min_years_exp} years of experience …")
        multipliers = _hard_filter_multipliers(df, min_years_exp)
        ensemble    = ensemble * multipliers

        df = df.copy()
        df["score_bge"]           = np.round(bge_s,    2)
        df["score_minilm"]        = np.round(mini_s,   2)
        df["score_cross_encoder"] = np.round(cross_s,  2)
        df["score_bm25"]          = np.round(bm25_s,   2)
        df["score_ner_skills"]    = np.round(ner_s,    2)
        df["score_temporal"]      = np.round(temp_s,   2)
        df["exp_multiplier"]      = np.round(multipliers, 2)
        df["match_score"]         = np.round(ensemble, 2)

        return df.sort_values("match_score", ascending=False)


# ══════════════════════════════════════════════════════════════════════════════
#  DISPLAY
# ══════════════════════════════════════════════════════════════════════════════

SCORE_COLS = [
    "score_bge", "score_minilm", "score_cross_encoder", "score_bm25",
    "score_ner_skills", "score_temporal", "exp_multiplier", "match_score",
]
SIGNAL_LABELS = {
    "score_bge":           f"BGE Bi-Encoder         ({int(WEIGHTS['bge']*100)}%)",
    "score_minilm":        f"MiniLM Bi-Encoder      ({int(WEIGHTS['minilm']*100)}%)",
    "score_cross_encoder": f"Cross-Encoder [DL]     ({int(WEIGHTS['cross_encoder']*100)}%)",
    "score_bm25":          f"BM25 Retrieval         ({int(WEIGHTS['bm25']*100)}%)",
    "score_ner_skills":    f"NER Skill Overlap      ({int(WEIGHTS['ner_skills']*100)}%)",
    "score_temporal":      f"Temporal Recency       ({int(WEIGHTS['temporal']*100)}%)",
    "exp_multiplier":      "Experience Multiplier  (pre-filter)",
    "match_score":         "FINAL SCORE",
}


def _banner(t):
    print("\n" + "═"*62)
    print(f"  {t}")
    print("═"*62)


def display_results(top_df: pd.DataFrame, job_position: str):
    _banner(f"TOP {len(top_df)} CANDIDATES — {job_position.upper()}")
    try:
        from IPython.display import display as ipy_display
        _ipython_display(top_df, job_position, ipy_display)
    except (ImportError, ModuleNotFoundError):
        _ascii_display(top_df)


def _ipython_display(top_df, job_position, ipy_display):
    display_df         = top_df[SCORE_COLS].copy()
    display_df.index   = [f"Rank #{i+1}" for i in range(len(display_df))]
    display_df.columns = [SIGNAL_LABELS.get(c, c) for c in display_df.columns]

    def _colour(val):
        if isinstance(val, float) and val <= 1.0:
            return "background-color: #ffe0e0; font-weight: bold; text-align:center" if val < 1.0 \
                   else "background-color: #f0f0f0; text-align:center"
        colour = "#c6efce" if val >= 75 else "#ffeb9c" if val >= 50 else "#ffc7ce"
        return f"background-color: {colour}; font-weight: bold; text-align: center"

    styled = (
        display_df.style
        .applymap(_colour)
        .set_caption(f"<b>ATS Results — {job_position}</b>")
        .set_table_styles([
            {"selector": "caption",
             "props": [("font-size","16px"),("text-align","left"),("padding","8px 0")]},
            {"selector": "th",
             "props": [("background-color","#2d3748"),("color","white"),
                       ("padding","6px 12px"),("font-size","12px")]},
            {"selector": "td",
             "props": [("padding","6px 14px"),("font-size","13px")]},
        ])
        .format("{:.2f}")
    )
    ipy_display(styled)

    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 3))
    ranks  = [f"Rank #{i+1}" for i in range(len(top_df))]
    scores = top_df["match_score"].values
    cols   = ["#276749" if s >= 75 else "#b7791f" if s >= 50 else "#c53030" for s in scores]
    bars   = ax.barh(ranks[::-1], scores[::-1], color=cols[::-1])
    ax.bar_label(bars, fmt="%.1f", padding=4, fontsize=11, fontweight="bold")
    ax.set_xlim(0, 105)
    ax.set_xlabel("ATS Match Score (0–100)")
    ax.set_title(f"Candidate Rankings — {job_position}", fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()


def _ascii_display(top_df):
    seps = {"score_cross_encoder", "score_ner_skills", "exp_multiplier", "match_score"}
    for rank, (_, row) in enumerate(top_df.iterrows(), start=1):
        print(f"\n{'─'*64}")
        print(f"  RANK #{rank}")
        snippet = row["combined_resume"].replace("\n", " | ")[:155]
        print(f"  {snippet}…\n")
        print(f"  ┌{'─'*42}┬{'─'*11}┐")
        print(f"  │ {'Signal':<40} │  {'Score':>7}  │")
        for col in SCORE_COLS:
            if col in seps:
                print(f"  ├{'─'*42}┼{'─'*11}┤")
            label  = SIGNAL_LABELS.get(col, col)
            value  = row[col]
            marker = "  ◄" if col == "match_score" else ""
            flag   = "  ⚠" if col == "exp_multiplier" and value < 1.0 else ""
            print(f"  │ {label:<40} │  {value:>6.2f}   │{marker}{flag}")
        print(f"  └{'─'*42}┴{'─'*11}┘")
    print("\n" + "═"*64)


# ══════════════════════════════════════════════════════════════════════════════
#  ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    scorer = IndustryATSScorer()

    _banner("📝  RECRUITER INPUT DASHBOARD")
    job_position        = input("1. Job Title (e.g., Senior Software Engineer): ")
    years_of_experience = input("2. Required Years of Experience (e.g., 5):     ")
    skillset            = input("3. Required Skills (e.g., Python, AWS, SQL):   ")
    job_description     = input("4. Brief Job Description:                       ")

    recruiter_query = (
        f"Title: {job_position}\n"
        f"Required Experience: {years_of_experience} years\n"
        f"Skills: {skillset}\n"
        f"Description: {job_description}"
    )

    KAGGLE_CANDIDATE_PATH = (
        "/kaggle/input/datasets/ckshetty/"
        "candidate-job-role-dataset/candidate_job_role_dataset.csv"
    )

    try:
        candidates_df = scorer.load_and_prep_candidates(
            KAGGLE_CANDIDATE_PATH, limit=1000
        )
    except FileNotFoundError:
        print(f"\n❌  File not found: {KAGGLE_CANDIDATE_PATH}")
        exit()

    top_n     = 5
    scored_df = scorer.find_top_candidates(
        recruiter_query,
        candidates_df,
        min_years_exp=int(years_of_experience or 0),
        top_n=top_n,
        run_diagnostic=True,    # ← set False once working correctly
    )

    display_results(scored_df.head(top_n), job_position)

In [ ]:
"""
Industry-Grade Ensemble ATS Scorer  (Bug-Fix Release)
======================================================
Fixes applied vs. previous version:

  BUG 1 — NER Skill Overlap always 0
    Cause : EntityRuler used plain lowercase string patterns, which spaCy
            matches case-sensitively at the token level. "Python" in a resume
            never matched the pattern "python".
    Fix   : Every pattern now uses the token-level dict format
            [{"LOWER": token}, …] which is case-insensitive by design.

  BUG 2 — Temporal Recency always 0
    Cause : Most ATS CSV datasets don't embed "2019–2022" date ranges inside
            free-text skill descriptions. When no year is found near a skill,
            the old code returned 0 — penalising candidates unfairly.
    Fix   : When no date context is found for a skill the score is 50.0
            (neutral / unknown) not 0. Only *confirmed* stale skills are
            penalised. Also detects "X years of experience" phrases as a
            proxy for recency.

  BUG 3 — Experience Multiplier always 0.50 (penalised)
    Cause : Regex r"(\\d{1,2})\\+?\\s*(?:years?|yrs?)" missed common formats
            such as "Experience: 5", "total experience 8 years", or a numeric
            CSV column named 'years_experience' / 'experience'.
    Fix   : (a) Broader regex covering more natural-language patterns.
            (b) Direct numeric-column scan of the original CSV row.
            (c) When no experience data is found at all, use 0.75 (soft
                neutral) instead of the full 0.50 penalty.

Scoring Architecture (unchanged)
─────────────────────────────────────────────────────────────────────
BUCKET          SIGNALS                         BUCKET WT  SIGNAL WT
──────────────  ──────────────────────────────  ─────────  ─────────
Semantic        BGE Bi-Encoder                    45 %       22.5 %
                MiniLM Bi-Encoder                            22.5 %
Precision       Cross-Encoder (DL, sigmoid)       45 %       25.0 %
                BM25 Okapi                                   20.0 %
Validation      NER Skill Overlap                 10 %        6.0 %
                Temporal Recency                              4.0 %

Pre-filter multiplier (applied after ensemble):
  years found >= required  →  × 1.00
  years found <  required  →  × 0.50
  years not detectable     →  × 0.75  (soft neutral — benefit of the doubt)
"""

import re
import string
import datetime
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity

try:
    from rank_bm25 import BM25Okapi
except ImportError:
    raise ImportError("pip install rank-bm25")

try:
    import spacy
except ImportError:
    raise ImportError("pip install spacy && python -m spacy download en_core_web_sm")


# ══════════════════════════════════════════════════════════════════════════════
#  CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

CURRENT_YEAR    = datetime.datetime.now().year
HALF_LIFE_YEARS = 5       # recency half-life in years
HARD_FILTER_PENALTY  = 0.50   # multiplier when years < required
SOFT_NEUTRAL_MULT    = 0.75   # multiplier when years can't be determined

WEIGHTS = {
    "bge":           0.225,
    "minilm":        0.225,
    "cross_encoder": 0.250,
    "bm25":          0.200,
    "ner_skills":    0.060,
    "temporal":      0.040,
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9


# ══════════════════════════════════════════════════════════════════════════════
#  SKILL SYNONYM TABLE
# ══════════════════════════════════════════════════════════════════════════════

SKILL_SYNONYMS: dict[str, list[str]] = {
    "python":                      ["py"],
    "javascript":                  ["js", "node.js", "nodejs", "node"],
    "typescript":                  ["ts"],
    "golang":                      ["go lang", "go"],
    "c++":                         ["cpp", "c plus plus"],
    "c#":                          ["csharp", "dotnet", ".net"],
    "ruby on rails":               ["rails", "ror"],
    "machine learning":            ["ml", "statistical learning"],
    "deep learning":               ["dl", "neural networks", "neural network"],
    "natural language processing": ["nlp", "text mining"],
    "computer vision":             ["cv", "image recognition"],
    "large language models":       ["llm", "llms", "generative ai", "gen ai"],
    "reinforcement learning":      ["rl"],
    "amazon web services":         ["aws", "amazon cloud"],
    "google cloud platform":       ["gcp", "google cloud"],
    "microsoft azure":             ["azure"],
    "kubernetes":                  ["k8s", "kube"],
    "docker":                      ["containerization", "containers"],
    "continuous integration":      ["ci", "ci/cd", "cicd"],
    "continuous deployment":       ["cd", "ci/cd", "cicd"],
    "infrastructure as code":      ["iac", "terraform", "pulumi"],
    "postgresql":                  ["postgres", "psql"],
    "microsoft sql server":        ["mssql", "sql server"],
    "mongodb":                     ["mongo"],
    "elasticsearch":               ["elastic", "es"],
    "apache kafka":                ["kafka"],
    "apache spark":                ["spark", "pyspark"],
    "react":                       ["react.js", "reactjs"],
    "angular":                     ["angular.js", "angularjs"],
    "vue":                         ["vue.js", "vuejs"],
    "graphql":                     ["graph ql"],
    "data science":                ["ds", "data scientist"],
    "data engineering":            ["data engineer", "de"],
    "business intelligence":       ["bi", "business analytics"],
    "etl":                         ["extract transform load", "data pipeline"],
    "agile":                       ["scrum", "kanban", "sprint"],
    "rest api":                    ["restful", "rest", "api development"],
    "microservices":               ["micro services", "soa"],
    "artificial intelligence":     ["ai"],
    "sql":                         ["structured query language"],
    "java":                        ["java ee", "java se", "spring", "spring boot"],
    "android":                     ["android development", "android sdk"],
    "ios":                         ["swift", "objective-c", "xcode"],
    "flutter":                     ["dart"],
    "devops":                      ["dev ops", "site reliability", "sre"],
}

_ALIAS_TO_CANONICAL: dict[str, str] = {
    alias: canonical
    for canonical, aliases in SKILL_SYNONYMS.items()
    for alias in aliases
}


# ══════════════════════════════════════════════════════════════════════════════
#  NER PIPELINE  (FIX 1 — case-insensitive token patterns)
# ══════════════════════════════════════════════════════════════════════════════

def _make_ner_patterns() -> list[dict]:
    """
    Build EntityRuler patterns using the token-level dict format:
        [{"LOWER": "machine"}, {"LOWER": "learning"}]
    spaCy applies LOWER matching case-insensitively, so "Python",
    "PYTHON", and "python" all match the same pattern.
    """
    patterns = []
    seen     = set()
    for canonical, aliases in SKILL_SYNONYMS.items():
        for surface in [canonical] + aliases:
            if surface in seen:
                continue
            seen.add(surface)
            tokens = surface.split()
            patterns.append({
                "label":   "TECH_SKILL",
                "pattern": [{"LOWER": t} for t in tokens],
            })
    return patterns


def _build_ner_pipeline():
    nlp   = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])
    ruler = nlp.add_pipe("entity_ruler", config={"overwrite_ents": True})
    ruler.add_patterns(_make_ner_patterns())
    return nlp


def _extract_ner_skills(nlp, text: str) -> set:
    doc = nlp(text[:50_000])
    return {ent.text.lower() for ent in doc.ents if ent.label_ == "TECH_SKILL"}


def _score_ner_skills(nlp, query_skills: set, texts: list) -> np.ndarray:
    if not query_skills:
        return np.full(len(texts), 50.0)
    scores = []
    for text in texts:
        resume_skills = _extract_ner_skills(nlp, text)
        hit = len(query_skills & resume_skills)
        scores.append((hit / len(query_skills)) * 100.0)
    return np.array(scores)


# ══════════════════════════════════════════════════════════════════════════════
#  STRUCTURED KEY-VALUE PARSER
#  The combined_resume field is in "col: value\ncol: value" format.
#  Parsing this directly is more reliable than regex-scanning free text.
# ══════════════════════════════════════════════════════════════════════════════

def _parse_kv(combined_resume: str) -> dict[str, str]:
    """
    Parses the 'col: value\\ncol: value' format produced by load_and_prep_candidates
    into a lowercase-key dictionary.

    Example input line:  "Years of Experience: 7"
    Example output:      {"years of experience": "7"}
    """
    kv = {}
    for line in combined_resume.split("\n"):
        if ":" in line:
            key, _, val = line.partition(":")
            kv[key.strip().lower()] = val.strip()
    return kv


# Keywords that identify an experience column
_EXP_KEYWORDS = ["experience", "years", "exp", "yrs", "tenure", "seniority"]

# Regex for extracting a numeric value from strings like "7", "7.5", "7+"
_NUM_RE = re.compile(r"(\d+(?:\.\d+)?)")

# Regex for free-text fallback (when KV parsing finds nothing)
_EXP_TEXT_PATTERNS = [
    re.compile(r"(\d{1,2})\+?\s*(?:years?|yrs?)\s+(?:of\s+)?experience", re.I),
    re.compile(r"experience\s*[:\-]?\s*(\d{1,2})", re.I),
    re.compile(r"over\s+(\d{1,2})\s+years?", re.I),
    re.compile(r"more\s+than\s+(\d{1,2})\s+years?", re.I),
    re.compile(r"(\d{1,2})\+?\s*(?:years?|yrs?)", re.I),
]


# ══════════════════════════════════════════════════════════════════════════════
#  TEMPORAL RECENCY — structured-data-aware
#
#  Root cause of "all 50.00":
#    The Kaggle dataset is a structured CSV, NOT a free-text resume.
#    There are no "Python (2019–2023)" strings to parse.
#
#  New approach (two-tier):
#    Tier 1 — KV column scan: look for a "skills" or "technology" column and
#             check whether each required skill is explicitly listed there.
#             Skills found in a dedicated column → confidently current (high score).
#             Skills absent from the column → likely not in toolkit (low score).
#    Tier 2 — Experience-year proxy: use years_of_experience to estimate
#             recency. A professional with 8 years of exp presumably used
#             their listed skills within the last few years.
# ══════════════════════════════════════════════════════════════════════════════

# Column name fragments that indicate a dedicated skills/tech column
_SKILL_COL_KEYWORDS = ["skill", "tech", "tool", "stack", "language", "framework",
                        "competenc", "proficien", "expertise"]


def _score_temporal(query_skills: set, texts: list) -> np.ndarray:
    """
    Structured-CSV temporal scoring.

    For each resume:
      1. Parse KV pairs.
      2. Find any 'skills'/'technology' column → check query skill coverage.
         coverage = fraction of required skills explicitly listed in that column.
         Mapped to [60, 100]: present = assumed current.
      3. Find 'years of experience' column → estimate activity recency.
         Professionals with ≥5 yrs likely still using their stack → boost toward 80.
      4. Blend both signals. If neither is available → 50.0 (neutral).
    """
    if not query_skills:
        return np.full(len(texts), 50.0)

    scores = []
    for text in texts:
        kv = _parse_kv(text)

        # ── Tier 1: skill-column coverage ────────────────────────────────────
        skill_coverage = None
        for key, val in kv.items():
            if any(kw in key for kw in _SKILL_COL_KEYWORDS):
                val_lower = val.lower()
                hit = sum(1 for s in query_skills if s in val_lower)
                skill_coverage = (hit / len(query_skills))   # 0.0 – 1.0
                break   # use the first matching skills column

        # ── Tier 2: experience-year proxy ────────────────────────────────────
        exp_years = None
        for key, val in kv.items():
            if any(kw in key for kw in _EXP_KEYWORDS):
                m = _NUM_RE.search(val)
                if m:
                    candidate_val = float(m.group(1))
                    if 0 < candidate_val < 60:
                        exp_years = candidate_val
                        break

        # ── Blend ─────────────────────────────────────────────────────────────
        if skill_coverage is not None and exp_years is not None:
            # Skills present in dedicated column + decent experience → high recency
            coverage_score = 60.0 + skill_coverage * 40.0   # 60–100
            exp_boost      = min(20.0, exp_years * 2.0)      # 0–20 bonus
            score          = min(100.0, coverage_score + exp_boost * (1 - skill_coverage))
        elif skill_coverage is not None:
            score = 60.0 + skill_coverage * 40.0
        elif exp_years is not None:
            # No skills column — use experience as a weak proxy
            # More years → likely more recent relevant activity
            score = min(85.0, 40.0 + exp_years * 3.5)
        else:
            score = 50.0   # truly unknown → neutral

        scores.append(score)

    return np.array(scores)


# ══════════════════════════════════════════════════════════════════════════════
#  HARD-REQUIREMENT PRE-FILTER — structured-data-aware
#
#  Root cause of "all 0.75":
#    _extract_max_years built a fake single-column Series and called
#    row.index — so the original CSV column names were never visible.
#    Also, the KV parser wasn't being used, so "Years of Experience: 7"
#    in combined_resume was missed entirely.
#
#  New approach:
#    Pass 0 — KV parse of combined_resume (catches "Years of Experience: 7").
#    Pass 1 — original CSV row column scan (catches numeric columns directly).
#    Pass 2 — free-text regex fallback.
# ══════════════════════════════════════════════════════════════════════════════

def _extract_max_years(row: pd.Series) -> int | None:
    """
    Three-pass experience year extraction.

    Pass 0 — Parse structured KV lines from combined_resume.
             Catches "Years of Experience: 7" or "Experience: Senior (8 yrs)".
    Pass 1 — Scan original CSV column values directly by column name.
    Pass 2 — Free-text regex fallback across the full resume string.
    """
    resume_text = str(row.get("combined_resume", ""))

    # ── Pass 0: KV parse (PRIMARY — most reliable for this dataset) ───────────
    kv = _parse_kv(resume_text)
    for key, val in kv.items():
        if any(kw in key for kw in _EXP_KEYWORDS):
            m = _NUM_RE.search(val)
            if m:
                candidate = float(m.group(1))
                if 0 < candidate < 60:
                    return int(candidate)

    # ── Pass 1: original CSV row column scan ──────────────────────────────────
    for col in row.index:
        col_norm = col.lower().replace(" ", "_").replace("-", "_")
        if any(kw in col_norm for kw in _EXP_KEYWORDS):
            try:
                val = float(row[col])
                if 0 < val < 60:
                    return int(val)
            except (ValueError, TypeError):
                # Non-numeric value in column — try extracting number from string
                m = _NUM_RE.search(str(row[col]))
                if m:
                    candidate = float(m.group(1))
                    if 0 < candidate < 60:
                        return int(candidate)

    # ── Pass 2: free-text regex ───────────────────────────────────────────────
    found = []
    for pattern in _EXP_TEXT_PATTERNS:
        for m in pattern.finditer(resume_text):
            try:
                found.append(int(float(m.group(1))))
            except (IndexError, ValueError):
                pass

    return max(found) if found else None


def _hard_filter_multipliers(df: pd.DataFrame, min_years: int) -> np.ndarray:
    """
    Returns per-candidate multiplier:
      years_found >= min_years  →  1.00  (pass)
      years_found <  min_years  →  0.50  (hard fail)
      years not detectable      →  0.75  (soft neutral)
    """
    multipliers = []
    detected    = {"pass": 0, "fail": 0, "unknown": 0}
    sample_shown = False

    for _, row in df.iterrows():
        max_yrs = _extract_max_years(row)

        # Print one example to confirm extraction is working
        if not sample_shown:
            print(f"  Sample experience extraction: {max_yrs} years detected")
            sample_shown = True

        if max_yrs is None:
            mult = SOFT_NEUTRAL_MULT
            detected["unknown"] += 1
        elif max_yrs >= min_years:
            mult = 1.00
            detected["pass"] += 1
        else:
            mult = HARD_FILTER_PENALTY
            detected["fail"] += 1
        multipliers.append(mult)

    print(
        f"  Pre-filter — ✅ pass: {detected['pass']}  "
        f"❌ fail: {detected['fail']}  ❓ unknown: {detected['unknown']}"
    )
    return np.array(multipliers)


# ══════════════════════════════════════════════════════════════════════════════
#  SHARED HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def _expand_text(text: str) -> str:
    t = text.lower()
    for alias, canonical in _ALIAS_TO_CANONICAL.items():
        t = re.sub(rf"\b{re.escape(alias)}\b", canonical, t)
    return t


def _tokenise(text: str) -> list[str]:
    expanded = _expand_text(text)
    return expanded.translate(str.maketrans("", "", string.punctuation)).split()


def _normalise(arr: np.ndarray) -> np.ndarray:
    lo, hi = float(arr.min()), float(arr.max())
    if hi == lo:
        return np.full(len(arr), 50.0)
    return ((arr - lo) / (hi - lo)) * 100.0


def _sigmoid_scale(logits: np.ndarray) -> np.ndarray:
    return (1.0 / (1.0 + np.exp(-logits.astype(float)))) * 100.0


# ══════════════════════════════════════════════════════════════════════════════
#  DIAGNOSTIC HELPER  (run this to debug zero scores)
# ══════════════════════════════════════════════════════════════════════════════

def diagnose(scorer, query: str, sample_resume: str):
    """
    Prints a full diagnostic for one resume.
    Usage:
        diagnose(scorer, recruiter_query, candidates_df.iloc[0]['combined_resume'])
    """
    print("\n" + "═"*62)
    print("  DIAGNOSTIC REPORT  (first candidate)")
    print("═"*62)

    # ── NER ───────────────────────────────────────────────────────────────────
    q_skills = _extract_ner_skills(scorer.nlp, query)
    r_skills = _extract_ner_skills(scorer.nlp, sample_resume)
    overlap  = q_skills & r_skills
    print(f"\n  NER query skills   : {sorted(q_skills) or '⚠ NONE — add to SKILL_SYNONYMS'}")
    print(f"  NER resume skills  : {sorted(r_skills) or '⚠ NONE'}")
    print(f"  NER overlap        : {sorted(overlap)  or '⚠ NONE'}")

    # ── KV parse ──────────────────────────────────────────────────────────────
    kv = _parse_kv(sample_resume)
    print(f"\n  Structured KV columns detected ({len(kv)} total):")
    for k, v in list(kv.items())[:12]:
        print(f"    '{k}' → '{v[:60]}'")
    if len(kv) > 12:
        print(f"    … and {len(kv)-12} more")

    # ── Experience ────────────────────────────────────────────────────────────
    fake_row = pd.Series({"combined_resume": sample_resume})
    yrs = _extract_max_years(fake_row)
    print(f"\n  Experience detected : {yrs} years  {'✅' if yrs else '⚠ not found — check column names above'}")

    # ── Temporal ──────────────────────────────────────────────────────────────
    temp_score = _score_temporal(q_skills, [sample_resume])[0]
    print(f"  Temporal score     : {temp_score:.1f}")

    # ── Skill columns ─────────────────────────────────────────────────────────
    skill_kv = {k: v for k, v in kv.items() if any(kw in k for kw in _SKILL_COL_KEYWORDS)}
    if skill_kv:
        print(f"\n  Skill columns found:")
        for k, v in skill_kv.items():
            print(f"    '{k}' → '{v[:80]}'")
    else:
        print("\n  ⚠ No dedicated skill columns found in KV pairs")

    print("\n" + "═"*62)


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN SCORER CLASS
# ══════════════════════════════════════════════════════════════════════════════

class IndustryATSScorer:

    def __init__(
        self,
        bge_model:   str = "BAAI/bge-small-en-v1.5",
        mini_model:  str = "sentence-transformers/all-MiniLM-L6-v2",
        cross_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2",
    ):
        _banner("INDUSTRY-GRADE ATS SCORER — initialising")

        print(f"\n  [1/4] BGE bi-encoder     : {bge_model}")
        self.bge    = SentenceTransformer(bge_model)

        print(f"  [2/4] MiniLM bi-encoder  : {mini_model}")
        self.minilm = SentenceTransformer(mini_model)

        print(f"  [3/4] Cross-Encoder (DL) : {cross_model}")
        self.cross  = CrossEncoder(
            cross_model, max_length=512, default_activation_function=None
        )

        print("  [4/4] spaCy NER pipeline (case-insensitive patterns) …")
        self.nlp = _build_ner_pipeline()

        print("\n  ✅  All components ready.\n")

    def load_and_prep_candidates(
        self, csv_path: str, limit: int = None
    ) -> pd.DataFrame:
        print(f"Loading candidates from: {csv_path}")
        df = pd.read_csv(csv_path).fillna("")
        df["combined_resume"] = df.apply(
            lambda row: "\n".join(f"{c}: {v}" for c, v in row.items()), axis=1
        )
        if limit:
            df = df.head(limit)

        # Print column names to help debug experience extraction
        exp_cols = [c for c in df.columns if c.lower().replace(" ", "_") in _EXP_COL_NAMES]
        print(f"Prepared {len(df)} profiles.  Experience columns detected: {exp_cols or 'none — will use regex'}\n")
        return df

    def _score_bge(self, query, texts):
        q = self.bge.encode(query)
        r = self.bge.encode(texts, show_progress_bar=True)
        return np.maximum(0.0, cosine_similarity([q], r)[0] * 100)

    def _score_minilm(self, query, texts):
        q = self.minilm.encode(query)
        r = self.minilm.encode(texts, show_progress_bar=True)
        return np.maximum(0.0, cosine_similarity([q], r)[0] * 100)

    def _score_cross_encoder(self, query, texts):
        pairs  = [(query, t) for t in texts]
        logits = self.cross.predict(
            pairs, batch_size=32, show_progress_bar=True, convert_to_numpy=True
        )
        return _sigmoid_scale(logits)

    def _score_bm25(self, query, texts):
        tokenised = [_tokenise(t) for t in texts]
        bm25      = BM25Okapi(tokenised, k1=1.5, b=0.75)
        raw       = np.array(bm25.get_scores(_tokenise(query)), dtype=float)
        return _normalise(raw)

    def find_top_candidates(
        self,
        recruiter_query: str,
        df: pd.DataFrame,
        min_years_exp: int = 0,
        top_n: int = 5,
        run_diagnostic: bool = False,
    ) -> pd.DataFrame:

        texts        = df["combined_resume"].tolist()
        query_skills = _extract_ner_skills(self.nlp, recruiter_query)
        print(f"  NER extracted query skills: {sorted(query_skills) or '⚠ none — check synonym table'}\n")

        if run_diagnostic and len(texts) > 0:
            diagnose(self, recruiter_query, texts[0])

        print("─" * 62)
        print("[Semantic  1/2]  BGE Bi-Encoder …")
        bge_s   = self._score_bge(recruiter_query, texts)

        print("\n[Semantic  2/2]  MiniLM Bi-Encoder …")
        mini_s  = self._score_minilm(recruiter_query, texts)

        print("\n[Precision 1/2]  Cross-Encoder (sigmoid) …")
        cross_s = self._score_cross_encoder(recruiter_query, texts)

        print("\n[Precision 2/2]  BM25 Retrieval …")
        bm25_s  = self._score_bm25(recruiter_query, texts)

        print("\n[Validation 1/2]  NER Skill Overlap …")
        ner_s   = _score_ner_skills(self.nlp, query_skills, texts)

        print("\n[Validation 2/2]  Temporal Recency …")
        temp_s  = _score_temporal(query_skills, texts)

        W        = WEIGHTS
        ensemble = (
            W["bge"]           * bge_s
            + W["minilm"]      * mini_s
            + W["cross_encoder"] * cross_s
            + W["bm25"]        * bm25_s
            + W["ner_skills"]  * ner_s
            + W["temporal"]    * temp_s
        )

        print(f"\n[Pre-filter]  Checking ≥ {min_years_exp} years of experience …")
        multipliers = _hard_filter_multipliers(df, min_years_exp)
        ensemble    = ensemble * multipliers

        df = df.copy()
        df["score_bge"]           = np.round(bge_s,    2)
        df["score_minilm"]        = np.round(mini_s,   2)
        df["score_cross_encoder"] = np.round(cross_s,  2)
        df["score_bm25"]          = np.round(bm25_s,   2)
        df["score_ner_skills"]    = np.round(ner_s,    2)
        df["score_temporal"]      = np.round(temp_s,   2)
        df["exp_multiplier"]      = np.round(multipliers, 2)
        df["match_score"]         = np.round(ensemble, 2)

        return df.sort_values("match_score", ascending=False)


# ══════════════════════════════════════════════════════════════════════════════
#  DISPLAY
# ══════════════════════════════════════════════════════════════════════════════

SCORE_COLS = [
    "score_bge", "score_minilm", "score_cross_encoder", "score_bm25",
    "score_ner_skills", "score_temporal", "exp_multiplier", "match_score",
]
# Column names commonly used in CSVs to represent experience
_EXP_COL_NAMES = ["years_of_experience", "experience", "total_experience", "yrs_exp", "experience_years"]

SIGNAL_LABELS = {
    "score_bge":           f"BGE Bi-Encoder         ({int(WEIGHTS['bge']*100)}%)",
    "score_minilm":        f"MiniLM Bi-Encoder      ({int(WEIGHTS['minilm']*100)}%)",
    "score_cross_encoder": f"Cross-Encoder [DL]     ({int(WEIGHTS['cross_encoder']*100)}%)",
    "score_bm25":          f"BM25 Retrieval         ({int(WEIGHTS['bm25']*100)}%)",
    "score_ner_skills":    f"NER Skill Overlap      ({int(WEIGHTS['ner_skills']*100)}%)",
    "score_temporal":      f"Temporal Recency       ({int(WEIGHTS['temporal']*100)}%)",
    "exp_multiplier":      "Experience Multiplier  (pre-filter)",
    "match_score":         "FINAL SCORE",
}


def _banner(t):
    print("\n" + "═"*62)
    print(f"  {t}")
    print("═"*62)


def display_results(top_df: pd.DataFrame, job_position: str):
    _banner(f"TOP {len(top_df)} CANDIDATES — {job_position.upper()}")
    try:
        from IPython.display import display as ipy_display
        _ipython_display(top_df, job_position, ipy_display)
    except (ImportError, ModuleNotFoundError):
        _ascii_display(top_df)


def _ipython_display(top_df, job_position, ipy_display):
    display_df         = top_df[SCORE_COLS].copy()
    display_df.index   = [f"Rank #{i+1}" for i in range(len(display_df))]
    display_df.columns = [SIGNAL_LABELS.get(c, c) for c in display_df.columns]

    def _colour(val):
        if isinstance(val, float) and val <= 1.0:
            return "background-color: #ffe0e0; font-weight: bold; text-align:center" if val < 1.0 \
                   else "background-color: #f0f0f0; text-align:center"
        colour = "#c6efce" if val >= 75 else "#ffeb9c" if val >= 50 else "#ffc7ce"
        return f"background-color: {colour}; font-weight: bold; text-align: center"

    styled = (
        display_df.style
        .applymap(_colour)
        .set_caption(f"<b>ATS Results — {job_position}</b>")
        .set_table_styles([
            {"selector": "caption",
             "props": [("font-size","16px"),("text-align","left"),("padding","8px 0")]},
            {"selector": "th",
             "props": [("background-color","#2d3748"),("color","white"),
                       ("padding","6px 12px"),("font-size","12px")]},
            {"selector": "td",
             "props": [("padding","6px 14px"),("font-size","13px")]},
        ])
        .format("{:.2f}")
    )
    ipy_display(styled)

    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 3))
    ranks  = [f"Rank #{i+1}" for i in range(len(top_df))]
    scores = top_df["match_score"].values
    cols   = ["#276749" if s >= 75 else "#b7791f" if s >= 50 else "#c53030" for s in scores]
    bars   = ax.barh(ranks[::-1], scores[::-1], color=cols[::-1])
    ax.bar_label(bars, fmt="%.1f", padding=4, fontsize=11, fontweight="bold")
    ax.set_xlim(0, 105)
    ax.set_xlabel("ATS Match Score (0–100)")
    ax.set_title(f"Candidate Rankings — {job_position}", fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()


def _ascii_display(top_df):
    seps = {"score_cross_encoder", "score_ner_skills", "exp_multiplier", "match_score"}
    for rank, (_, row) in enumerate(top_df.iterrows(), start=1):
        print(f"\n{'─'*64}")
        print(f"  RANK #{rank}")
        snippet = row["combined_resume"].replace("\n", " | ")[:155]
        print(f"  {snippet}…\n")
        print(f"  ┌{'─'*42}┬{'─'*11}┐")
        print(f"  │ {'Signal':<40} │  {'Score':>7}  │")
        for col in SCORE_COLS:
            if col in seps:
                print(f"  ├{'─'*42}┼{'─'*11}┤")
            label  = SIGNAL_LABELS.get(col, col)
            value  = row[col]
            marker = "  ◄" if col == "match_score" else ""
            flag   = "  ⚠" if col == "exp_multiplier" and value < 1.0 else ""
            print(f"  │ {label:<40} │  {value:>6.2f}   │{marker}{flag}")
        print(f"  └{'─'*42}┴{'─'*11}┘")
    print("\n" + "═"*64)


# ══════════════════════════════════════════════════════════════════════════════
#  ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    scorer = IndustryATSScorer()

    _banner("📝  RECRUITER INPUT DASHBOARD")
    job_position        = input("1. Job Title (e.g., Senior Software Engineer): ")
    years_of_experience = input("2. Required Years of Experience (e.g., 5):     ")
    skillset            = input("3. Required Skills (e.g., Python, AWS, SQL):   ")
    job_description     = input("4. Brief Job Description:                       ")

    recruiter_query = (
        f"Title: {job_position}\n"
        f"Required Experience: {years_of_experience} years\n"
        f"Skills: {skillset}\n"
        f"Description: {job_description}"
    )

    KAGGLE_CANDIDATE_PATH = (
        "/kaggle/input/datasets/ckshetty/"
        "candidate-job-role-dataset/candidate_job_role_dataset.csv"
    )

    try:
        candidates_df = scorer.load_and_prep_candidates(
            KAGGLE_CANDIDATE_PATH, limit=1000
        )
    except FileNotFoundError:
        print(f"\n❌  File not found: {KAGGLE_CANDIDATE_PATH}")
        exit()

    top_n     = 5
    scored_df = scorer.find_top_candidates(
        recruiter_query,
        candidates_df,
        min_years_exp=int(years_of_experience or 0),
        top_n=top_n,
        run_diagnostic=True,    # ← set False once working correctly
    )

    display_results(scored_df.head(top_n), job_position)

In [ ]:
pip install rank-bm25
